[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabrielbelo2007/RP_PIGuard/blob/main/Notebooks/train.ipynb)

# PIGuard - Treinamento de Modelos na GPU G4 (Google Colab)

Este notebook serve como o disparador de execução remota dos treinamentos do modelo PIGuard utilizando aceleração por GPU no Google Colab.

### Três Cenários de Treinamento e Ablação Comparativa:
1. **`train_augmented`**: Treinado com o primeiro dataset aumentado (75.973 amostras com o conjunto sintético original do artigo).
2. **`train_new_augmented`**: Treinado com as novas 435 injeções geradas via gpt-4o-mini-2024-07-18 com funil de 4 filtros em 17 formatos estruturados.
3. **`train_cleaned`**: Treinado com a base limpa baseline (75.973 amostras sem nenhuma injeção aumentada da cauda longa).

Ao final de cada treinamento, os melhores pesos (`best_model.pth`) e logs são persistidos no Google Drive, seguidos pelo **Token-wise Recheck** para extração de trigger tokens e encerramento automático da VM para economia de créditos de GPU.

In [ ]:
# Célula 1: Verificação da GPU
!nvidia-smi

import torch
print("\n--- Informações de Hardware e PyTorch ---")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


### Setup: Google Drive, Repositório e Dependências
1. Monta o Google Drive para persistência definitiva de modelos e logs.
2. Clona ou atualiza o repositório `RP_PIGuard` na VM.
3. Instala dependências necessárias.
4. Define controle de retreinamento inteligente (`FORCE_RETRAIN`).

In [ ]:
# Célula 2: Montagem do Google Drive, Repositório e Dependências
import os
import sys
import shutil

# 1. Montagem do Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/RP_PIGuard_outputs"
os.makedirs(f"{DRIVE_OUTPUT_DIR}/train_augmented", exist_ok=True)
os.makedirs(f"{DRIVE_OUTPUT_DIR}/train_new_augmented", exist_ok=True)
os.makedirs(f"{DRIVE_OUTPUT_DIR}/train_cleaned", exist_ok=True)
print(f"Diretórios de persistência configurados no Google Drive: {DRIVE_OUTPUT_DIR}")

# 2. Clonagem ou sincronização do repositório
REPO_DIR = "/content/RP_PIGuard"
if not os.path.exists(REPO_DIR):
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass

    if not token and os.path.exists("/content/drive/MyDrive/RP_PIGuard"):
        print("Copiando repositório existente do Google Drive para o SSD local da VM...")
        shutil.copytree("/content/drive/MyDrive/RP_PIGuard", REPO_DIR)
    else:
        repo_url = f"https://{token}@github.com/gabrielbelo2007/RP_PIGuard.git" if token else "https://github.com/gabrielbelo2007/RP_PIGuard.git"
        print(f"Clonando repositório em {REPO_DIR}...")
        !git clone {repo_url} {REPO_DIR}
else:
    print(f"Repositório já presente em {REPO_DIR}. Atualizando branch main...")
    !cd {REPO_DIR} && git pull origin main || true

# 3. Mudar para a raiz do repositório
%cd {REPO_DIR}

# 4. Instalar dependências necessárias
!pip install -q "transformers>=4.44.0,<5.0.0" "datasets>=2.21.0" sentencepiece protobuf tiktoken pyyaml nltk tqdm accelerate

# 5. Controle de retreinamento:
# Se False, pula treinamentos cujos modelos (best_model.pth) já existam no Google Drive,
# poupando até 1h30 de computação desnecessária de GPU!
FORCE_RETRAIN = False
print(f"\nFORCE_RETRAIN configurado como: {FORCE_RETRAIN}")

# Adiciona ao sys.path e altera cwd explicitamente
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


### Treinamento 1: `datasets/train_augmented.json`
Treina o modelo PIGuard utilizando o primeiro dataset aumentado (75.973 amostras com o conjunto sintético original).
- Hiperparâmetros: `--batch_size 32`, `--eval_batch_size 32`, `--save_step 100`, `--epochs 3`.

In [ ]:
# Célula 3: Treinamento 1 - datasets/train_augmented.json
import os
import shutil

drive_target_1 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_augmented"
drive_best_1 = f"{drive_target_1}/best_model.pth"

if os.path.exists(drive_best_1) and not FORCE_RETRAIN:
    print(f"[SKIP] Modelo 1 já existe no Google Drive em: {drive_best_1}")
    print("Pulando Treinamento 1 para economizar recursos de GPU. (Defina FORCE_RETRAIN=True para reexecutar).")
else:
    !mkdir -p logs/train_augmented checkpoints/train_augmented
    !python train.py \
        --name train_augmented \
        --train_set datasets/train_augmented.json \
        --valid_set datasets/valid.json \
        --dataset_root datasets \
        --checkpoint_path checkpoints/train_augmented \
        --logs logs/train_augmented \
        --batch_size 32 \
        --eval_batch_size 32 \
        --save_step 100 \
        --epochs 3

    local_best_1 = "checkpoints/train_augmented/best_model.pth"
    local_log_1 = "logs/train_augmented/log_train_augmented.txt"
    if os.path.exists(local_best_1):
        print(f"\nCopiando {local_best_1} para {drive_target_1}...")
        shutil.copy(local_best_1, f"{drive_target_1}/best_model.pth")
        if os.path.exists(local_log_1):
            shutil.copy(local_log_1, f"{drive_target_1}/log_train_augmented.txt")
        print(f"Treinamento 1 salvo com sucesso no Google Drive: {drive_target_1}/best_model.pth")
        for fname in os.listdir("checkpoints/train_augmented"):
            if fname.startswith("epoch_") and fname.endswith(".pth"):
                os.remove(os.path.join("checkpoints/train_augmented", fname))
    else:
        print(f"Aviso: {local_best_1} não encontrado.")


### Treinamento 2: `datasets/train_new_augmented.json`
Treina o modelo PIGuard utilizando o segundo dataset aumentado (75.973 amostras com as novas injeções gpt-4o-mini-2024-07-18 em 17 formatos estruturados).
- Hiperparâmetros: `--batch_size 32`, `--eval_batch_size 32`, `--save_step 100`, `--epochs 3`.

In [ ]:
# Célula 4: Treinamento 2 - datasets/train_new_augmented.json
import os
import shutil

drive_target_2 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_new_augmented"
drive_best_2 = f"{drive_target_2}/best_model.pth"

if os.path.exists(drive_best_2) and not FORCE_RETRAIN:
    print(f"[SKIP] Modelo 2 já existe no Google Drive em: {drive_best_2}")
    print("Pulando Treinamento 2 para economizar recursos de GPU. (Defina FORCE_RETRAIN=True para reexecutar).")
else:
    !mkdir -p logs/train_new_augmented checkpoints/train_new_augmented
    !python train.py \
        --name train_new_augmented \
        --train_set datasets/train_new_augmented.json \
        --valid_set datasets/valid.json \
        --dataset_root datasets \
        --checkpoint_path checkpoints/train_new_augmented \
        --logs logs/train_new_augmented \
        --batch_size 32 \
        --eval_batch_size 32 \
        --save_step 100 \
        --epochs 3

    local_best_2 = "checkpoints/train_new_augmented/best_model.pth"
    local_log_2 = "logs/train_new_augmented/log_train_new_augmented.txt"
    if os.path.exists(local_best_2):
        print(f"\nCopiando {local_best_2} para {drive_target_2}...")
        shutil.copy(local_best_2, f"{drive_target_2}/best_model.pth")
        if os.path.exists(local_log_2):
            shutil.copy(local_log_2, f"{drive_target_2}/log_train_new_augmented.txt")
        print(f"Treinamento 2 salvo com sucesso no Google Drive: {drive_target_2}/best_model.pth")
        for fname in os.listdir("checkpoints/train_new_augmented"):
            if fname.startswith("epoch_") and fname.endswith(".pth"):
                os.remove(os.path.join("checkpoints/train_new_augmented", fname))
    else:
        print(f"Aviso: {local_best_2} não encontrado.")


### Treinamento 3: `datasets/train_cleaned.json` (Baseline Limpo)
Treina o modelo PIGuard utilizando a base limpa (75.973 amostras sem nenhuma injeção aumentada da cauda longa).
- Serve como a referência de ablação (*baseline*) para verificar o nível de over-defense natural do modelo antes do aumento de dados.
- Hiperparâmetros: `--batch_size 32`, `--eval_batch_size 32`, `--save_step 100`, `--epochs 3`.

In [ ]:
# Célula 5: Treinamento 3 - datasets/train_cleaned.json (Baseline Limpo)
import os
import shutil

drive_target_3 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_cleaned"
drive_best_3 = f"{drive_target_3}/best_model.pth"

if os.path.exists(drive_best_3) and not FORCE_RETRAIN:
    print(f"[SKIP] Modelo 3 já existe no Google Drive em: {drive_best_3}")
    print("Pulando Treinamento 3 para economizar recursos de GPU. (Defina FORCE_RETRAIN=True para reexecutar).")
else:
    !mkdir -p logs/train_cleaned checkpoints/train_cleaned
    !python train.py \
        --name train_cleaned \
        --train_set datasets/train_cleaned.json \
        --valid_set datasets/valid.json \
        --dataset_root datasets \
        --checkpoint_path checkpoints/train_cleaned \
        --logs logs/train_cleaned \
        --batch_size 32 \
        --eval_batch_size 32 \
        --save_step 100 \
        --epochs 3

    local_best_3 = "checkpoints/train_cleaned/best_model.pth"
    local_log_3 = "logs/train_cleaned/log_train_cleaned.txt"
    if os.path.exists(local_best_3):
        print(f"\nCopiando {local_best_3} para {drive_target_3}...")
        shutil.copy(local_best_3, f"{drive_target_3}/best_model.pth")
        if os.path.exists(local_log_3):
            shutil.copy(local_log_3, f"{drive_target_3}/log_train_cleaned.txt")
        print(f"Treinamento 3 salvo com sucesso no Google Drive: {drive_target_3}/best_model.pth")
        for fname in os.listdir("checkpoints/train_cleaned"):
            if fname.startswith("epoch_") and fname.endswith(".pth"):
                os.remove(os.path.join("checkpoints/train_cleaned", fname))
    else:
        print(f"Aviso: {local_best_3} não encontrado.")


### Token-wise Recheck (Seção 4.2 do artigo PIGuard)
Execução da extração das *trigger words* identificadas por cada um dos 3 modelos treinados utilizando a GPU G4.
Testa todos os 128.000 tokens do vocabulário do DeBERTa de forma isolada (`[CLS, token, SEP]`) para detectar viés de atalhos (*shortcut bias*).

In [ ]:
# Célula 6: Token-wise Recheck - Modelo 1 (train_augmented)
import os

model_path_1 = "checkpoints/train_augmented/best_model.pth"
if not os.path.exists(model_path_1):
    model_path_1 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_augmented/best_model.pth"

output_path_1 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_augmented/trigger_words_train_augmented.json"

if os.path.exists(model_path_1):
    print(f"Disparando Token-wise Recheck para o modelo 1: {model_path_1}")
    !python token_recheck.py \
        --checkpoint {model_path_1} \
        --output {output_path_1} \
        --batch_size 256
else:
    print(f"Aviso: Checkpoint {model_path_1} não encontrado.")


In [ ]:
# Célula 7: Token-wise Recheck - Modelo 2 (train_new_augmented)
import os

model_path_2 = "checkpoints/train_new_augmented/best_model.pth"
if not os.path.exists(model_path_2):
    model_path_2 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_new_augmented/best_model.pth"

output_path_2 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_new_augmented/trigger_words_train_new_augmented.json"

if os.path.exists(model_path_2):
    print(f"Disparando Token-wise Recheck para o modelo 2: {model_path_2}")
    !python token_recheck.py \
        --checkpoint {model_path_2} \
        --output {output_path_2} \
        --batch_size 256
else:
    print(f"Aviso: Checkpoint {model_path_2} não encontrado.")


In [ ]:
# Célula 8: Token-wise Recheck - Modelo 3 (train_cleaned - Baseline Limpo)
import os

model_path_3 = "checkpoints/train_cleaned/best_model.pth"
if not os.path.exists(model_path_3):
    model_path_3 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_cleaned/best_model.pth"

output_path_3 = "/content/drive/MyDrive/RP_PIGuard_outputs/train_cleaned/trigger_words_train_cleaned.json"

if os.path.exists(model_path_3):
    print(f"Disparando Token-wise Recheck para o modelo 3: {model_path_3}")
    !python token_recheck.py \
        --checkpoint {model_path_3} \
        --output {output_path_3} \
        --batch_size 256
else:
    print(f"Aviso: Checkpoint {model_path_3} não encontrado.")


### Desligamento Automático do Runtime (Google Colab)
Libera imediatamente os recursos de computação (GPU G4) desconectando e encerrando o ambiente via `runtime.unassign()`.
Isso evita consumo acidental de créditos/horas de GPU após a conclusão de todas as etapas.

In [ ]:
# Célula 9: Desligamento Automático da VM no Google Colab
import time
import os

print("\n" + "=" * 70)
print("🎉 Processamento concluído com sucesso!")
print("📁 Artefatos persistidos com segurança no Google Drive:")
for exp in ["train_augmented", "train_new_augmented", "train_cleaned"]:
    path = f"/content/drive/MyDrive/RP_PIGuard_outputs/{exp}/best_model.pth"
    status = "OK (Presente)" if os.path.exists(path) else "Pendente/Ausente"
    print(f"   - [{status}] {path}")
print("=" * 70)
print("⚡ Desconectando o runtime da VM para evitar consumo desnecessário de computação...")

try:
    from google.colab import runtime
    time.sleep(5)
    runtime.unassign()
except Exception as e:
    print(f"Não foi possível desconectar automaticamente via runtime.unassign(): {e}")
